#Model 1: Power Consumption Prediction

##Import Data

In [ ]:
pip install ucimlrepo

In [ ]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
individual_household_electric_power_consumption = fetch_ucirepo(id=235)

# data (as pandas dataframes)
data = individual_household_electric_power_consumption.data.features


/usr/local/lib/python3.11/dist-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


In [ ]:
data.head(5)

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   object 
 1   Time                   object 
 2   Global_active_power    object 
 3   Global_reactive_power  object 
 4   Voltage                object 
 5   Global_intensity       object 
 6   Sub_metering_1         object 
 7   Sub_metering_2         object 
 8   Sub_metering_3         float64
dtypes: float64(1), object(8)
memory usage: 142.5+ MB


##Data Preprocessing

In [ ]:
import pandas as pd

data['datetime'] = pd.to_datetime(data['Date'] + ' ' + data['Time'], format='%d/%m/%Y %H:%M:%S')

data['hour'] = data['datetime'].dt.hour
data['day'] = data['datetime'].dt.day
data['month'] = data['datetime'].dt.month
data['weekday'] = data['datetime'].dt.weekday  # 0 = Monday, 6 = Sunday
data = data[data['datetime'].dt.year == 2010]

In [ ]:
# List of columns to convert
numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity',
                'Sub_metering_1', 'Sub_metering_2']

# Convert them to numeric
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')


<ipython-input-6-eebba471c061>:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data[col] = pd.to_numeric(data[col], errors='coerce')


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 475023 entries, 1600236 to 2075258
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   Date                   475023 non-null  object        
 1   Time                   475023 non-null  object        
 2   Global_active_power    457394 non-null  float64       
 3   Global_reactive_power  457394 non-null  float64       
 4   Voltage                457394 non-null  float64       
 5   Global_intensity       457394 non-null  float64       
 6   Sub_metering_1         457394 non-null  float64       
 7   Sub_metering_2         457394 non-null  float64       
 8   Sub_metering_3         457394 non-null  float64       
 9   datetime               475023 non-null  datetime64[ns]
 10  hour                   475023 non-null  int32         
 11  day                    475023 non-null  int32         
 12  month                  475023 non-null  in

##Data Cleaning

In [ ]:
data.isnull().sum()

,0
Date,0
Time,0
Global_active_power,17629
Global_reactive_power,17629
Voltage,17629
Global_intensity,17629
Sub_metering_1,17629
Sub_metering_2,17629
Sub_metering_3,17629
datetime,0


In [ ]:
data_clean = data.dropna()


In [ ]:
data_clean.isnull().sum()

,0
Date,0
Time,0
Global_active_power,0
Global_reactive_power,0
Voltage,0
Global_intensity,0
Sub_metering_1,0
Sub_metering_2,0
Sub_metering_3,0
datetime,0


In [ ]:
data_clean.shape

(457394, 14)

##Define features and target

In [ ]:
features = ['Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3',
            'Global_reactive_power', 'Voltage',
            'hour', 'day', 'weekday', 'month']

X = data_clean[features]
y = data_clean['Global_active_power']


##Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


##Modelling with Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model_rf = RandomForestRegressor(random_state=42)
model_rf.fit(X_train, y_train)
y_pred_rf = model_rf.predict(X_test)


## Model Evaluation

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

results = {

    "Random Forest": {
        "MSE": mean_squared_error(y_test, y_pred_rf),
        "R2": r2_score(y_test, y_pred_rf)
    }

}

import pandas as pd
pd.DataFrame(results).T


,MSE,R2
Random Forest,0.000533,0.99939


#Model 2 - Time Based High Energy Usage Prediction

##Import Data

In [8]:
pip install ucimlrepo

In [9]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
individual_household_electric_power_consumption = fetch_ucirepo(id=235)

# data (as pandas dataframes)
data = individual_household_electric_power_consumption.data.features


/usr/local/lib/python3.11/dist-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


In [10]:
data.head(5)

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   object 
 1   Time                   object 
 2   Global_active_power    object 
 3   Global_reactive_power  object 
 4   Voltage                object 
 5   Global_intensity       object 
 6   Sub_metering_1         object 
 7   Sub_metering_2         object 
 8   Sub_metering_3         float64
dtypes: float64(1), object(8)
memory usage: 142.5+ MB


##Data Preprocessing

In [13]:
data['datetime'] = pd.to_datetime(data['Date'] + ' ' + data['Time'], format='%d/%m/%Y %H:%M:%S')

data['hour'] = data['datetime'].dt.hour
data['day'] = data['datetime'].dt.day
data['month'] = data['datetime'].dt.month
data['weekday'] = data['datetime'].dt.weekday


##Define High Energy Usage Column

In [14]:
import pandas as pd

# Example threshold: values above 75th percentile = high
data['Global_active_power'] = pd.to_numeric(data['Global_active_power'], errors='coerce')
threshold = data['Global_active_power'].quantile(0.75)

data['high_usage'] = (data['Global_active_power'] > threshold).astype(int)



##Define Target and Features

In [15]:
time_features = ['hour', 'day', 'month', 'weekday']
X = data[time_features]
y = data['high_usage']


##Train-Test Split

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)


##Modelling with Random Forest and Evaluation

In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

clf = RandomForestClassifier()
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.92      0.95      0.94    312698
           1       0.83      0.76      0.80    102354

    accuracy                           0.90    415052
   macro avg       0.88      0.86      0.87    415052
weighted avg       0.90      0.90      0.90    415052

